### 1. Import Dependencies

In [1]:
## !pip install tensorflow

In [2]:
## !pip install opencv-python

In [3]:
##!pip uninstall -y mediapipe numpy


##!pip cache purge

## !pip install numpy mediapipe --no-cache-dir --force-reinstall

print(
    "Installed latest compatible versions. Please restart the runtime (Runtime -> Restart Runtime) before running code")

Installed latest compatible versions. Please restart the runtime (Runtime -> Restart Runtime) before running code


In [4]:
import numpy as np
import os
from matplotlib import pyplot as plt
import time

import cv2
import mediapipe as mp
from mediapipe.tasks.python import vision
from mediapipe.tasks import python


### 2. Keypoints using MP Holistic

In [5]:
def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Color conversion
    image.flags.writeable = False  # Image is no longer writable
    results = model.process(image)  # Make prediction
    image.flags.writeable = True  # Image is now writeable
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)  # Color conversion
    return image, results

In [6]:
# ===== CONNECTIONS =====
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (5, 9), (9, 10), (10, 11), (11, 12),
    (9, 13), (13, 14), (14, 15), (15, 16),
    (13, 17), (17, 18), (18, 19), (19, 20),
    (0, 17)
]

POSE_CONNECTIONS = [
    (11, 13), (13, 15),
    (12, 14), (14, 16),
    (11, 12),
    (11, 23), (12, 24),
    (23, 24),
    (23, 25), (25, 27),
    (24, 26), (26, 28)
]
FACE_OVAL = [
    (10, 338), (338, 297), (297, 332), (332, 284), (284, 251), (251, 389),
    (389, 356), (356, 454), (454, 323), (323, 361), (361, 288), (288, 397),
    (397, 365), (365, 379), (379, 378), (378, 400), (400, 377), (377, 152),
    (152, 148), (148, 176), (176, 149), (149, 150), (150, 136), (136, 172),
    (172, 58), (58, 132), (132, 93), (93, 234), (234, 127), (127, 162),
    (162, 21), (21, 54), (54, 103), (103, 67), (67, 109), (109, 10)
]
LIPS = [
    (61, 146), (146, 91), (91, 181), (181, 84), (84, 17), (17, 314), (314, 405),
    (405, 321), (321, 375), (375, 291), (291, 61),
    (78, 95), (95, 88), (88, 178), (178, 87), (87, 14), (14, 317), (317, 402),
    (402, 318), (318, 324), (324, 308), (308, 78)
]
LEFT_EYE = [
    (33, 7), (7, 163), (163, 144), (144, 145), (145, 153), (153, 154),
    (154, 155), (155, 133), (133, 33)
]
RIGHT_EYE = [
    (362, 382), (382, 381), (381, 380), (380, 374), (374, 373), (373, 390),
    (390, 249), (249, 263), (263, 362)
]

In [7]:
def draw_styled_landmarks(image, pose_result=None, hand_result=None, face_result=None):
    h, w, _ = image.shape
    overlay = image.copy()

    # ===== FACE =====
    if face_result and face_result.face_landmarks:
        for face in face_result.face_landmarks:

            # --- FACE OVAL ---
            for c in FACE_OVAL:
                p1, p2 = face[c[0]], face[c[1]]
                x1, y1 = int(p1.x * w), int(p1.y * h)
                x2, y2 = int(p2.x * w), int(p2.y * h)
                cv2.line(overlay, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # --- LIPS ---
            for c in LIPS:
                p1, p2 = face[c[0]], face[c[1]]
                x1, y1 = int(p1.x * w), int(p1.y * h)
                x2, y2 = int(p2.x * w), int(p2.y * h)
                cv2.line(overlay, (x1, y1), (x2, y2), (0, 0, 255), 2)

            # --- EYES ---
            for c in LEFT_EYE:
                p1, p2 = face[c[0]], face[c[1]]
                x1, y1 = int(p1.x * w), int(p1.y * h)
                x2, y2 = int(p2.x * w), int(p2.y * h)
                cv2.line(overlay, (x1, y1), (x2, y2), (255, 255, 0), 1)

            for c in RIGHT_EYE:
                p1, p2 = face[c[0]], face[c[1]]
                x1, y1 = int(p1.x * w), int(p1.y * h)
                x2, y2 = int(p2.x * w), int(p2.y * h)
                cv2.line(overlay, (x1, y1), (x2, y2), (255, 255, 0), 1)

            # --- точки (лёгкие) ---
            for lm in face:
                x, y = int(lm.x * w), int(lm.y * h)
                cv2.circle(overlay, (x, y), 1, (200, 200, 200), -1)

    # ===== POSE =====
    if pose_result and pose_result.pose_landmarks:
        for pose in pose_result.pose_landmarks:

            # линии
            for c in POSE_CONNECTIONS:
                p1, p2 = pose[c[0]], pose[c[1]]
                x1, y1 = int(p1.x * w), int(p1.y * h)
                x2, y2 = int(p2.x * w), int(p2.y * h)
                cv2.line(overlay, (x1, y1), (x2, y2), (80, 44, 121), 2)

            # точки
            for lm in pose:
                x, y = int(lm.x * w), int(lm.y * h)
                cv2.circle(overlay, (x, y), 4, (80, 22, 10), -1)

    # ===== HANDS =====
    if hand_result and hand_result.hand_landmarks:
        for i, hand in enumerate(hand_result.hand_landmarks):

            # определяем левая/правая (если есть handedness)
            is_left = False
            if hand_result.handedness:
                label = hand_result.handedness[i][0].category_name
                is_left = (label == "Left")

            # цвета как в твоём коде
            if is_left:
                point_color = (121, 22, 76)
                line_color = (121, 44, 250)
            else:
                point_color = (245, 117, 66)
                line_color = (245, 66, 230)

            # линии
            for c in HAND_CONNECTIONS:
                p1, p2 = hand[c[0]], hand[c[1]]
                x1, y1 = int(p1.x * w), int(p1.y * h)
                x2, y2 = int(p2.x * w), int(p2.y * h)
                cv2.line(overlay, (x1, y1), (x2, y2), line_color, 2)

            # точки
            for lm in hand:
                x, y = int(lm.x * w), int(lm.y * h)
                cv2.circle(overlay, (x, y), 4, point_color, -1)

    # ===== BLEND =====
    image = cv2.addWeighted(overlay, 0.6, image, 0.4, 0)
    return image

In [8]:
def draw_overlay(image, pose_result=None, hand_result=None, face_result=None):
    h, w, _ = image.shape

    # создаём слой
    overlay = image.copy()

    # ===== POSE =====
    if pose_result and pose_result.pose_landmarks:
        for pose in pose_result.pose_landmarks:

            # линии
            for c in POSE_CONNECTIONS:
                p1, p2 = pose[c[0]], pose[c[1]]
                x1, y1 = int(p1.x * w), int(p1.y * h)
                x2, y2 = int(p2.x * w), int(p2.y * h)
                cv2.line(overlay, (x1, y1), (x2, y2), (80, 200, 255), 3)

            # точки
            for lm in pose:
                x, y = int(lm.x * w), int(lm.y * h)
                cv2.circle(overlay, (x, y), 5, (0, 255, 255), -1)

    # ===== HANDS =====
    if hand_result and hand_result.hand_landmarks:
        for i, hand in enumerate(hand_result.hand_landmarks):

            # определяем руку
            is_left = False
            if hand_result.handedness:
                label = hand_result.handedness[i][0].category_name
                is_left = (label == "Left")

            # цвета
            if is_left:
                line_color = (255, 100, 255)
                point_color = (255, 0, 200)
            else:
                line_color = (0, 255, 100)
                point_color = (0, 200, 255)

            # линии
            for c in HAND_CONNECTIONS:
                p1, p2 = hand[c[0]], hand[c[1]]
                x1, y1 = int(p1.x * w), int(p1.y * h)
                x2, y2 = int(p2.x * w), int(p2.y * h)
                cv2.line(overlay, (x1, y1), (x2, y2), line_color, 3)

            # точки
            for lm in hand:
                x, y = int(lm.x * w), int(lm.y * h)
                cv2.circle(overlay, (x, y), 6, point_color, -1)

    # ===== FACE (soft overlay) =====
    if face_result and face_result.face_landmarks:
        for face in face_result.face_landmarks:
            for lm in face:
                x, y = int(lm.x * w), int(lm.y * h)
                cv2.circle(overlay, (x, y), 1, (200, 200, 200), -1)

    # ===== BLEND (магия красоты) =====
    alpha = 0.6
    image = cv2.addWeighted(overlay, alpha, image, 1 - alpha, 0)

    return image

In [9]:
cap = cv2.VideoCapture(0)

pose_landmark = "pose_landmarker_lite.task"
hand_landmark = "hand_landmarker.task"
face_landmark = "face_landmarker.task"

# ===== POSE =====
pose_base = python.BaseOptions(model_asset_path=pose_landmark)
pose_options = vision.PoseLandmarkerOptions(
    base_options=pose_base,
    running_mode=vision.RunningMode.VIDEO
)
pose_detector = vision.PoseLandmarker.create_from_options(pose_options)

# ===== HANDS =====
hand_base = python.BaseOptions(model_asset_path=hand_landmark)
hand_options = vision.HandLandmarkerOptions(
    base_options=hand_base,
    running_mode=vision.RunningMode.VIDEO,
    num_hands=2
)
hand_detector = vision.HandLandmarker.create_from_options(hand_options)

# ===== FACE =====
face_base = python.BaseOptions(model_asset_path=face_landmark)
face_options = vision.FaceLandmarkerOptions(
    base_options=face_base,
    running_mode=vision.RunningMode.VIDEO
)
face_detector = vision.FaceLandmarker.create_from_options(face_options)

timestamp = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=frame
    )

    # ===== DETECTIONS =====
    pose_result = pose_detector.detect_for_video(mp_image, timestamp)
    hand_result = hand_detector.detect_for_video(mp_image, timestamp)
    face_result = face_detector.detect_for_video(mp_image, timestamp)

    timestamp += 1

    # draw landmarks
    frame = draw_overlay(frame, pose_result, hand_result, face_result)
    cv2.imshow('Feed', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

### 3. Extract Keypoint Values

In [10]:
def extract_keypoints(pose_result, hand_result, face_result):
    # POSE
    pose = np.array([
        [res.x, res.y, res.z, res.visibility]
        for res in pose_result.pose_landmarks[0]
    ]).flatten() if pose_result.pose_landmarks else np.zeros(33 * 4)

    # HANDS
    lh = np.zeros(21 * 3)
    rh = np.zeros(21 * 3)

    if hand_result.hand_landmarks:
        for idx, hand_landmarks in enumerate(hand_result.hand_landmarks):
            label = hand_result.handedness[idx][0].category_name

            coords = np.array([
                [lm.x, lm.y, lm.z]
                for lm in hand_landmarks
            ]).flatten()

            if label == "Left":
                lh = coords
            elif label == "Right":
                rh = coords

    #FACE
    face = np.array([
        [lm.x, lm.y, lm.z]
        for lm in face_result.face_landmarks[0]
    ]).flatten() if face_result.face_landmarks else np.zeros(478 * 3)

    return np.concatenate((pose, face, lh, rh))


### 4. Setup Folders for Collection

In [11]:
# Path for exported data
DATA_PATH = os.path.join("MP_Data")

# Actions that we try to detect
actions = np.array(['hello', 'thanks', 'iloveyou'])

# Thirty videos worth of data
no_sequences = 30

# Videos are going to be 30 frames in legth
sequence_length = 30

# для каждого слова мы создадим по папку и в них будут 30 кадров

In [12]:
import os
import numpy as np

DATA_PATH = os.path.join('MP_Data')
actions = np.array(['hello', 'thanks', 'iloveyou'])
no_sequences = 30

print("DATA_PATH:", os.path.abspath(DATA_PATH))
print("actions:", actions)
print("no_sequences:", no_sequences)

for action in actions:
    for sequence in range(no_sequences):
        path = os.path.join(DATA_PATH, action, str(sequence))
        os.makedirs(path, exist_ok=True)
        print(f"Создана папка: {path}")

print("Готово!")



DATA_PATH: C:\Users\admin\PycharmProjects\HandTalker\src\live_time_sign language\MP_Data
actions: ['hello' 'thanks' 'iloveyou']
no_sequences: 30
Создана папка: MP_Data\hello\0
Создана папка: MP_Data\hello\1
Создана папка: MP_Data\hello\2
Создана папка: MP_Data\hello\3
Создана папка: MP_Data\hello\4
Создана папка: MP_Data\hello\5
Создана папка: MP_Data\hello\6
Создана папка: MP_Data\hello\7
Создана папка: MP_Data\hello\8
Создана папка: MP_Data\hello\9
Создана папка: MP_Data\hello\10
Создана папка: MP_Data\hello\11
Создана папка: MP_Data\hello\12
Создана папка: MP_Data\hello\13
Создана папка: MP_Data\hello\14
Создана папка: MP_Data\hello\15
Создана папка: MP_Data\hello\16
Создана папка: MP_Data\hello\17
Создана папка: MP_Data\hello\18
Создана папка: MP_Data\hello\19
Создана папка: MP_Data\hello\20
Создана папка: MP_Data\hello\21
Создана папка: MP_Data\hello\22
Создана папка: MP_Data\hello\23
Создана папка: MP_Data\hello\24
Создана папка: MP_Data\hello\25
Создана папка: MP_Data\hello\26
С

# 5. Collect Keypoint Values for Training and Testing

In [13]:
cap = cv2.VideoCapture(0)

pose_landmark = "pose_landmarker_lite.task"
hand_landmark = "hand_landmarker.task"
face_landmark = "face_landmarker.task"

# ===== POSE =====
pose_base = python.BaseOptions(model_asset_path=pose_landmark)
pose_options = vision.PoseLandmarkerOptions(
    base_options=pose_base,
    running_mode=vision.RunningMode.VIDEO
)
pose_detector = vision.PoseLandmarker.create_from_options(pose_options)

# ===== HANDS =====
hand_base = python.BaseOptions(model_asset_path=hand_landmark)
hand_options = vision.HandLandmarkerOptions(
    base_options=hand_base,
    running_mode=vision.RunningMode.VIDEO,
    num_hands=2
)
hand_detector = vision.HandLandmarker.create_from_options(hand_options)

# ===== FACE =====
face_base = python.BaseOptions(model_asset_path=face_landmark)
face_options = vision.FaceLandmarkerOptions(
    base_options=face_base,
    running_mode=vision.RunningMode.VIDEO
)
face_detector = vision.FaceLandmarker.create_from_options(face_options)

timestamp = 0

# Loop through actions
for action in actions:
    # Loop through sequences aka videos
    for sequence in range(no_sequences):
        # Loop through video length aka sequence length
        for frame_num in range(sequence_length):
            # read feed
            ret, frame = cap.read()
            if not ret:
                break

            mp_image = mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=frame
            )

            # ===== DETECTIONS =====
            pose_result = pose_detector.detect_for_video(mp_image, timestamp)
            hand_result = hand_detector.detect_for_video(mp_image, timestamp)
            face_result = face_detector.detect_for_video(mp_image, timestamp)

            timestamp += 1

            # draw landmarks
            frame = draw_styled_landmarks(frame, pose_result, hand_result, face_result)

            # Apply collection logic
            if frame_num % 2 == 0:
                cv2.putText(frame, 'STARTING COLLECTION', (120, 200),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 4, cv2.LINE_AA)
                cv2.putText(frame, 'Collecting frames for {} Video Number {}'.format(action, sequence), (15, 12),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)
                # Show to screen
                cv2.imshow('OpenCV Feed', frame)
                cv2.waitKey(2000)
            else:
                cv2.putText(frame, 'Collecting frames for {} Video Number {}'.format(action, sequence), (15, 12),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)
                # Show to screen
                cv2.imshow('OpenCV Feed', frame)

            keypoints = extract_keypoints(pose_result, hand_result, face_result)
            npy_path = os.path.join(DATA_PATH, action, str(sequence), str(frame_num))
            np.save(npy_path, keypoints)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

In [ ]:
# todo top method optimization

### 6. Preprocess Data and Create Labels and Featureds

In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

In [16]:
label_map = {label:num for num, label in enumerate(actions)}
label_map

{np.str_('hello'): 0, np.str_('thanks'): 1, np.str_('iloveyou'): 2}

In [18]:
sequences, labels = [], []
for action in actions:
    for sequence in range(no_sequences):
        window = []
        for frame_num in range(sequence_length):
            res = np.load(os.path.join(DATA_PATH, action, str(sequence), "{}.npy".format(frame_num)))
            window.append(res)
        sequences.append(window)
        labels.append(label_map[action])

FileNotFoundError: [Errno 2] No such file or directory: 'MP_Data\\hello\\0\\0.npy'

In [17]:
np.array(sequences).shape


NameError: name 'sequences' is not defined

In [ ]:
X = np.array(sequences)

In [ ]:
X.shape

In [ ]:
y = to_categorical(labels).astype(int)
y

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.05)
y_test.shape

### 7. Build and Train LSTM Neural Network

In [19]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import TensorBoard

In [ ]:
log_dir = os.path.join('Logs')
tb_callback = TensorBoard(log_dir=log_dir)